Q1: Fundamental differences between DDL, DML, and DQL
Answer:-
1. DDL (Data Definition Language): Defines database schema.
Example: CREATE TABLE Students (ID INT PRIMARY KEY, Name VARCHAR(50));
2. DML (Data Manipulation Language): Manages data inside tables.
Example: INSERT INTO Students VALUES (1, 'Alice');
3. DQL (Data Query Language): Retrieves data.
Example:SELECT * FROM Students;

Q2: Purpose of SQL Constraints
Answer:-
- PRIMARY KEY: Ensures uniqueness (e.g., CustomerID).
- FOREIGN KEY: Maintains relationships (e.g., Orders.CustomerID → Customers.CustomerID).
- UNIQUE: Prevents duplicates (e.g., Email must be unique).

Q3: LIMIT vs OFFSET
Answer:-
1. LIMIT: Restricts rows returned.
2. OFFSET: Skips rows.
Retrieve page 3 (records 21–30):
SELECT * FROM Products
LIMIT 10 OFFSET 20;

Q4: Common Table Expression (CTE)
Answer:-
CTE: Temporary result set for readability and recursion.
Example:
WITH ExpensiveProducts AS (
  SELECT ProductName, Price FROM Products WHERE Price > 500
)
SELECT * FROM ExpensiveProducts;

Q5: SQL Normalization
Answer:-
- Goal: Reduce redundancy, improve integrity.
- 1NF: Atomic values, no repeating groups.
- 2NF: 1NF + no partial dependency.
- 3NF: 2NF + no transitive dependency.


In [ ]:
#Q6: Create ECommerceDB
#Answer:-
CREATE DATABASE ECommerceDB;
USE ECommerceDB;

CREATE TABLE Categories (
  CategoryID INT PRIMARY KEY,
  CategoryName VARCHAR(50) NOT NULL UNIQUE
);

CREATE TABLE Products (
  ProductID INT PRIMARY KEY,
  ProductName VARCHAR(100) NOT NULL UNIQUE,
  CategoryID INT,
  Price DECIMAL(10,2) NOT NULL,
  StockQuantity INT,
  FOREIGN KEY (CategoryID) REFERENCES Categories(CategoryID)
);

CREATE TABLE Customers (
  CustomerID INT PRIMARY KEY,
  CustomerName VARCHAR(100) NOT NULL,
  Email VARCHAR(100) UNIQUE,
  JoinDate DATE
);

CREATE TABLE Orders (
  OrderID INT PRIMARY KEY,
  CustomerID INT,
  OrderDate DATE NOT NULL,
  TotalAmount DECIMAL(10,2),
  FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID)
);

-- Insert statements for Categories, Products, Customers, Orders



In [ ]:
#Q7: Customer Orders Report
#Answer:-
SELECT c.CustomerName, c.Email,
       COUNT(o.OrderID) AS TotalNumberOfOrders
FROM Customers c
LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerName, c.Email
ORDER BY c.CustomerName;


In [ ]:
#Q8: Product Info with Category
#Answer:-
SELECT p.ProductName, p.Price, p.StockQuantity, c.CategoryName
FROM Products p
JOIN Categories c ON p.CategoryID = c.CategoryID
ORDER BY c.CategoryName, p.ProductName;


In [ ]:
#Q9: CTE + Window Function
#Answer:-
WITH RankedProducts AS (
  SELECT c.CategoryName, p.ProductName, p.Price,
         ROW_NUMBER() OVER (PARTITION BY c.CategoryName ORDER BY p.Price DESC) AS rn
  FROM Products p
  JOIN Categories c ON p.CategoryID = c.CategoryID
)
SELECT CategoryName, ProductName, Price
FROM RankedProducts
WHERE rn <= 2;


In [ ]:
#Q10: Sakila Database Analysis
#Answer:-
# 1. Top 5 customers by spend:
SELECT c.first_name, c.last_name, c.email, SUM(p.amount) AS total_spent
FROM customer c
JOIN payment p ON c.customer_id = p.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 5;


In [ ]:
# 2. Top 3 categories by rentals:
SELECT cat.name, COUNT(r.rental_id) AS rental_count
FROM rental r
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film_category fc ON i.film_id = fc.film_id
JOIN category cat ON fc.category_id = cat.category_id
GROUP BY cat.name
ORDER BY rental_count DESC
LIMIT 3;


In [ ]:
# 3.Films per store & never rented:
SELECT s.store_id,
       COUNT(i.inventory_id) AS total_films,
       SUM(CASE WHEN r.rental_id IS NULL THEN 1 ELSE 0 END) AS never_rented
FROM store s
JOIN inventory i ON s.store_id = i.store_id
LEFT JOIN rental r ON i.inventory_id = r.inventory_id
GROUP BY s.store_id;


In [ ]:
# 4.Revenue per month (2023):
SELECT MONTH(p.payment_date) AS month, SUM(p.amount) AS revenue
FROM payment p
WHERE YEAR(p.payment_date) = 2023
GROUP BY MONTH(p.payment_date)
ORDER BY month;


In [ ]:
# 5.Customers with >10 rentals in last 6 months:
SELECT c.customer_id, c.first_name, c.last_name, COUNT(r.rental_id) AS rental_count
FROM customer c
JOIN rental r ON c.customer_id = r.customer_id
WHERE r.rental_date >= DATE_SUB(CURDATE(), INTERVAL 6 MONTH)
GROUP BY c.customer_id
HAVING rental_count > 10;
